In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import warnings
import random
warnings.filterwarnings("ignore")

In [2]:
def second_degree(p):
    return 1.1*p - 0.5*p**2


def fourth_degree(p):
    return -p**4 + 22*p**3 - 165*p**2 + 480*p-150


def exponential(p):
    return 100*np.exp((-((p-5) ** 2))/20)

In [3]:
# Create a dictionary mapping strings to functions
function_mapping = {
    "second": second_degree,
    "fourth": fourth_degree,
    "exponential": exponential
}

degree_mapping = {
    "second": 2,
    "fourth": 4,
    'exponential': 4
}

bounds_mapping = {
    "second": [0., 2.2],
    "fourth": [0.36, 10.5],
    "exponential": [0.0, 10.0]
}

noise_mapping = {
    "second": 0.1,
    "fourth": 0.1,
    "exponential": 3
}

In [4]:
from scipy.optimize import minimize_scalar

# Define the negative of the fourth_degree function to find the maximum


def neg_fourth_degree(p):
    return -fourth_degree(p)


# Use minimize_scalar to find the value of p that minimizes the negative fourth_degree function
result = minimize_scalar(neg_fourth_degree, bounds=(0, 10), method='bounded')

# The optimal value of p
optimal_p_fourth = result.x
optimal_p_second = 1.1
optimal_p_exponent = 5

In [5]:
SIZE = 100

actual_revenue_function="fourth"
model_revenue_function="fourth"

In [6]:
if actual_revenue_function == "second":
    optimal = np.array([second_degree(optimal_p_second) for _ in range(SIZE)])
elif actual_revenue_function == "fourth":
    optimal = np.array([fourth_degree(optimal_p_fourth) for _ in range(SIZE)])
elif actual_revenue_function == "exponential":
    optimal = np.array([exponential(optimal_p_exponent) for _ in range(SIZE)])

In [7]:
def Constrained_Iterated_Least_Squares(k,actual_revenue_function="fourth",model_revenue_function="fourth"):
    data = []
    final = []
    p = np.random.uniform(
        bounds_mapping[model_revenue_function][0], bounds_mapping[model_revenue_function][1])
    y = function_mapping[actual_revenue_function](
        p)+np.random.normal(0, noise_mapping[actual_revenue_function])
    data.append([p, y])
    final.append(y)
    for i in tqdm(range(1, SIZE)):
        df = pd.DataFrame(data, columns=['p', 'y'])
        
        coeffs = np.polyfit(
            df['p'], df['y'], degree_mapping[model_revenue_function])

        # Define the polynomial function
        poly = np.poly1d(coeffs)

        poly_derivative = np.polyder(poly)
        critical_points = np.roots(poly_derivative)

        # critical_points = np.append(
        #     critical_points, bounds_mapping[model_revenue_function])

        # Find the value of x that maximizes the polynomial
        x = np.real(critical_points)
        y = np.real(poly(x))
        average_price = np.mean(df['p'])
        last_p = df['p'].iloc[-1]
        delta = last_p-average_price
        if (abs(delta) < k*(i**-0.25)):
            price = average_price+np.sign(delta)*k*(i**-0.25)
        else:
            price = x[np.argmax(y)]
        final.append(function_mapping[actual_revenue_function](
            price))
        data.append([price, function_mapping[actual_revenue_function](
            price)+np.random.normal(0, noise_mapping[actual_revenue_function])])

    return final

In [8]:
results=Constrained_Iterated_Least_Squares(0.5)

100%|██████████| 99/99 [00:00<00:00, 983.93it/s]


In [10]:
import json

data_total=[]
for actual_revenue_function in tqdm(["second", "fourth", "exponential"]):
    for model_revenue_function in ["second", "fourth", "exponential"]:
        best_till_now_thmp = []
        revenue = []
        for i in range(100):
            rets = Constrained_Iterated_Least_Squares(0.5,actual_revenue_function=actual_revenue_function, model_revenue_function=model_revenue_function)
            revenue.append(rets)
        data = {
            "actual_dim": actual_revenue_function,
            "model_dim": model_revenue_function,
            "data": [list(revenue) for revenue in revenue],
        }
        data_total.append(data)
with open('data.json', 'w') as f:
    json.dump(data_total, f)


100%|██████████| 3/3 [00:38<00:00, 12.89s/it]
